# ViT-B/16 + MIL com Optuna (Ordinal Focal Loss)

Treina Vision Transformer (ViT-B/16) com:
- **MIL (Gated Attention pooling)** sobre bags de patches histopatológicos
- **BCEFocalOrdinalLoss** combinando focal loss + penalidade ordinal
- **Optuna** para busca dos pesos ótimos da perda (`gamma`, `w_focal`, `w_ord`)

In [ ]:
import sys
sys.path.append('../../../')

import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
from torch.utils.data import DataLoader
from torch.utils.data.sampler import RandomSampler, SequentialSampler
from torchvision.models import vit_b_16, ViT_B_16_Weights
import albumentations as Albu
from warmup_scheduler import GradualWarmupScheduler
from sklearn.metrics import accuracy_score, cohen_kappa_score, f1_score, recall_score, precision_score
from tqdm import tqdm
import optuna
from optuna.pruners import MedianPruner
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

from utils.dataset import PandasWithMilDataset
from utils.mil import ViTMIL

## Configuração

In [ ]:
# Reprodutibilidade
SEED = 42
torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Hiperparâmetros fixos
BATCH_SIZE      = 8
NUM_WORKERS     = 8
OUTPUT_CLASSES  = 5
INIT_LR         = 3e-5
WEIGHT_DECAY    = 1e-4
WARMUP_FACTOR   = 2
WARMUP_EPOCHS   = 2
N_EPOCHS        = 50
DROPOUT_RATE    = 0.4
PATIENCE        = 12
FINE_TUNE       = 100
MAX_PATCHES     = 49
GRAD_CLIP       = 1.0

AMP_DTYPE = torch.bfloat16

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# Optuna — subset de dados + menos patches para reduzir custo por trial (~19× mais rápido)
# sem cache: backprop completo preservado, backbone continua sendo atualizado
OPTUNA_BATCH_SIZE  = 4
OPTUNA_MAX_PATCHES = 16    # patches por bag no Optuna (era 49); reduz forwards/batch em ~3×
OPTUNA_SUBSET_FRAC = 0.25  # fração do train usada no Optuna; reduz batches/época em 4×
N_OPTUNA_TRIALS    = 30
N_OPTUNA_EPOCHS    = 5     # épocas por trial (era 8)

# Caminhos
ROOT_DIR   = '../../..'
IMAGES_DIR = '/home/woshington/Projects/Doutorado/bag_of_patches'

os.makedirs('logs',   exist_ok=True)
os.makedirs('models', exist_ok=True)

MODEL_PATH = 'models/vit-base-mil-optuna.pth'
LOG_PATH   = 'logs/vit-base-mil-optuna.txt'

print(f'BATCH_SIZE={BATCH_SIZE} | MAX_PATCHES={MAX_PATCHES} | NUM_WORKERS={NUM_WORKERS}')
print(f'AMP dtype: {AMP_DTYPE} | GRAD_CLIP={GRAD_CLIP} | PATIENCE={PATIENCE}')
print(f'Optuna: SUBSET={OPTUNA_SUBSET_FRAC:.0%} | MAX_PATCHES={OPTUNA_MAX_PATCHES} | EPOCHS={N_OPTUNA_EPOCHS}')

## Função de Perda

In [3]:
class BCEFocalOrdinalLoss(nn.Module):
    """
    Perda combinada: Focal Loss + Penalidade Ordinal.

    loss = w_focal * focal + w_ord * ordinal_mse

    Args:
        gamma:   expoente focal (controla foco em exemplos difíceis).
        w_focal: peso do termo focal.
        w_ord:   peso do termo ordinal (MSE sobre classe esperada).
    """

    def __init__(self, gamma: float = 2.0, w_focal: float = 1.0, w_ord: float = 1.0):
        super().__init__()
        self.gamma   = gamma
        self.w_focal = w_focal
        self.w_ord   = w_ord

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        targets = targets.to(logits.device).float()
        probs   = torch.sigmoid(logits)

        # Focal loss
        bce        = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        p_t        = probs * targets + (1 - probs) * (1 - targets)
        focal_loss = ((1 - p_t) ** self.gamma * bce).mean()

        # Penalidade ordinal
        expected_cls = probs.sum(dim=1)
        target_cls   = targets.sum(dim=1)
        max_cls      = logits.shape[1]
        ord_loss     = ((expected_cls - target_cls) ** 2).mean() / (max_cls ** 2)

        return self.w_focal * focal_loss + self.w_ord * ord_loss


print('BCEFocalOrdinalLoss definida.')

BCEFocalOrdinalLoss definida.


## Carregamento de Dados

In [4]:
def remove_nonexistent(df, images_dir):
    mask = df['image_id'].apply(lambda x: os.path.isdir(os.path.join(images_dir, x)))
    return df[mask].reset_index(drop=True)


df_all = pd.read_csv(f'{ROOT_DIR}/data/train_5fold.csv')
df_all.columns = df_all.columns.str.strip()

# Filtragem por entropia (remove 20% mais difíceis)
df_entropy = pd.read_csv(f'{ROOT_DIR}/data/entropy.csv')
df_entropy = df_entropy.sort_values('difficulty_score', ascending=False)
n_remove   = int(len(df_entropy) * 0.2)
ids_remove = set(df_entropy.head(n_remove)['image_id'])
df_all     = df_all[~df_all['image_id'].isin(ids_remove)].reset_index(drop=True)

train_idx = np.where(df_all['fold'] != 3)[0]
valid_idx = np.where(df_all['fold'] == 3)[0]

df_train = df_all.loc[train_idx].reset_index(drop=True)
df_val   = df_all.loc[valid_idx].reset_index(drop=True)
df_test  = pd.read_csv(f'{ROOT_DIR}/data/test.csv')
df_test.columns = df_test.columns.str.strip()

df_train = remove_nonexistent(df_train, IMAGES_DIR)
df_val   = remove_nonexistent(df_val,   IMAGES_DIR)
df_test  = remove_nonexistent(df_test,  IMAGES_DIR)

print(f'Train: {len(df_train)} | Val: {len(df_val)} | Test: {len(df_test)}')
print('Distribuição de classes (train):')
print(df_train['isup_grade'].value_counts().sort_index())

Train: 7073 | Val: 1767 | Test: 1590
Distribuição de classes (train):
isup_grade
0    1953
1    1802
2     899
3     826
4     832
5     761
Name: count, dtype: int64


## Augmentação

> ViT-B/16 espera imagens **224×224** com normalização ImageNet.
> `normalize=False` nos Datasets pois `Albu.Normalize` já faz a conversão correta.

In [5]:
train_transforms = Albu.Compose([
    Albu.Transpose(p=0.5),
    Albu.VerticalFlip(p=0.5),
    Albu.HorizontalFlip(p=0.5),
    Albu.RandomRotate90(p=0.5),
    # Simula variações de coloração histológica entre lâminas/laboratórios
    # Albu.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.15, hue=0.05, p=0.4),
    Albu.Resize(224, 224),
    # ImageNet normalization — obrigatório para pesos pré-treinados do ViT
    Albu.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

val_transforms = Albu.Compose([
    Albu.Resize(224, 224),
    Albu.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

## Funções de Treino e Validação

In [6]:
def training_step(model, dataloader, optimizer, device, loss_fn, scaler, grad_clip=1.0):
    model.train()
    losses = []
    for bag, mask, targets, _ in tqdm(dataloader):
        bag     = bag.to(device, non_blocking=True)
        mask    = mask.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type='cuda', dtype=AMP_DTYPE):
            out  = model(bag, mask)
            loss = loss_fn(out['logits'], targets)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        scaler.step(optimizer)
        scaler.update()
        losses.append(loss.detach().cpu().item())
    return losses


def validation_step(model, dataloader, device, loss_fn):
    model.eval()
    val_loss, all_preds, all_targets = [], [], []
    with torch.no_grad():
        for bag, mask, targets, _ in tqdm(dataloader):
            bag     = bag.to(device, non_blocking=True)
            mask    = mask.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)
            with torch.autocast(device_type='cuda', dtype=AMP_DTYPE):
                out  = model(bag, mask)
                loss = loss_fn(out['logits'], targets)
            probs       = torch.sigmoid(out['logits'])
            preds       = (probs > 0.5).sum(dim=1)
            targets_cls = targets.sum(dim=1).long()
            all_preds.append(preds.cpu())
            all_targets.append(targets_cls.cpu())
            val_loss.append(loss.cpu().item())
    all_preds   = torch.cat(all_preds).numpy()
    all_targets = torch.cat(all_targets).numpy()
    return {
        'val_loss':      np.mean(val_loss),
        'val_acc':       accuracy_score(all_targets, all_preds),
        'val_kappa':     cohen_kappa_score(all_targets, all_preds, weights='quadratic'),
        'val_f1':        f1_score(all_targets, all_preds, average='macro', zero_division=0),
        'val_recall':    recall_score(all_targets, all_preds, average='macro', zero_division=0),
        'val_precision': precision_score(all_targets, all_preds, average='macro', zero_division=0),
    }

## Busca de Hiperparâmetros com Optuna

O Optuna busca simultaneamente:
- **Loss**: `gamma`, `w_focal`, `w_ord` da `BCEFocalOrdinalLoss`
- **Otimizador**: `lr` (log-scale) e `weight_decay`

Treina por `N_OPTUNA_EPOCHS` épocas com `MedianPruner` e maximiza o **Kappa quadrático** de validação.

In [ ]:
# Subset aleatório fixo para o Optuna (OPTUNA_SUBSET_FRAC do train)
rng_optuna = np.random.default_rng(SEED)
optuna_idx = rng_optuna.choice(len(df_train), size=int(len(df_train) * OPTUNA_SUBSET_FRAC), replace=False)
df_optuna  = df_train.iloc[optuna_idx].reset_index(drop=True)

# Usa OPTUNA_MAX_PATCHES patches por bag — reduz forwards pelo backbone em ~3×
optuna_train_ds = PandasWithMilDataset(
    IMAGES_DIR, df_optuna, transforms=train_transforms,
    normalize=False, max_patches=OPTUNA_MAX_PATCHES
)
optuna_val_ds = PandasWithMilDataset(
    IMAGES_DIR, df_val, transforms=val_transforms,
    normalize=False, max_patches=OPTUNA_MAX_PATCHES
)

optuna_train_loader = DataLoader(
    optuna_train_ds, batch_size=OPTUNA_BATCH_SIZE, num_workers=NUM_WORKERS,
    sampler=RandomSampler(optuna_train_ds),
    pin_memory=True, drop_last=True,
)
optuna_val_loader = DataLoader(
    optuna_val_ds, batch_size=OPTUNA_BATCH_SIZE * 2, num_workers=NUM_WORKERS,
    sampler=SequentialSampler(optuna_val_ds),
    pin_memory=True,
)

print(f'Optuna subset: {len(df_optuna)} imgs ({OPTUNA_SUBSET_FRAC:.0%} do train) | {OPTUNA_MAX_PATCHES} patches/bag')
print(f'Train batches: {len(optuna_train_loader)} | Val batches: {len(optuna_val_loader)}')

In [ ]:
import types
from torch.utils.checkpoint import checkpoint as grad_checkpoint


def _checkpointed_encoder_forward(self, input: torch.Tensor) -> torch.Tensor:
    """Gradient checkpointing por bloco — troca ~40% de VRAM por recompute no backward."""
    x = input
    for layer in self.layers:
        x = grad_checkpoint(layer, x, use_reentrant=False)
    return self.ln(x)


def build_model(enable_checkpointing: bool = True):
    backbone = vit_b_16(weights=ViT_B_16_Weights.DEFAULT)
    if enable_checkpointing:
        backbone.encoder.forward = types.MethodType(
            _checkpointed_encoder_forward, backbone.encoder
        )
    return ViTMIL(
        model=backbone,
        output_classes=OUTPUT_CLASSES,
        fine_tune=FINE_TUNE,
        dropout_rate=DROPOUT_RATE,
        hidden_dim=512,
        gated=True,
        pool='att',
    ).to(device)


def objective(trial: optuna.Trial) -> float:
    gamma        = trial.suggest_float('gamma',        0.5, 3.0)
    w_focal      = trial.suggest_float('w_focal',      0.1, 2.0)
    w_ord        = trial.suggest_float('w_ord',        0.1, 2.0)
    lr           = trial.suggest_float('lr',           1e-5, 1e-4, log=True)
    weight_decay = trial.suggest_float('weight_decay', 1e-5, 1e-2, log=True)

    # Gradient checkpointing ativo: mantém VRAM sob controle com OPTUNA_BATCH_SIZE
    model   = build_model(enable_checkpointing=True)
    loss_fn = BCEFocalOrdinalLoss(gamma=gamma, w_focal=w_focal, w_ord=w_ord)
    opt     = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scaler  = torch.amp.GradScaler()

    best_kappa = 0.0
    for epoch in range(N_OPTUNA_EPOCHS):
        training_step(model, optuna_train_loader, opt, device, loss_fn, scaler, grad_clip=GRAD_CLIP)
        metrics = validation_step(model, optuna_val_loader, device, loss_fn)
        kappa   = metrics['val_kappa']

        print(f"[Trial {trial.number}, Epoch {epoch}] Kappa: {kappa:.4f}")

        trial.report(kappa, epoch)
        if trial.should_prune():
            del model
            torch.cuda.empty_cache()
            raise optuna.exceptions.TrialPruned()

        best_kappa = max(best_kappa, kappa)

    del model
    torch.cuda.empty_cache()
    return best_kappa


print('build_model com gradient checkpointing definido.')
print('Objective Optuna definido.')

In [ ]:
STUDY_DB   = 'sqlite:///logs/vit-base-mil-optuna.db'
STUDY_NAME = 'vit-base-mil-ordinal-focal'

optuna.logging.set_verbosity(optuna.logging.WARNING)

study = optuna.create_study(
    study_name=STUDY_NAME,
    direction='maximize',    
    storage=STUDY_DB,
    load_if_exists=True,
)

study.optimize(objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True) 

best = study.best_trial
print(f'\nMelhor trial: #{best.number}')
print(f'  Kappa: {best.value:.4f}')
print(f'  gamma:   {best.params["gamma"]:.4f}')
print(f'  w_focal: {best.params["w_focal"]:.4f}')
print(f'  w_ord:   {best.params["w_ord"]:.4f}')

  0%|          | 0/30 [00:00<?, ?it/s]

100%|██████████| 221/221 [04:20<00:00,  1.18s/it]


[Trial 1, Epoch 0] Kappa: 0.7888


100%|██████████| 221/221 [04:11<00:00,  1.14s/it]


[Trial 1, Epoch 1] Kappa: 0.7972


...

100%|██████████| 221/221 [03:56<00:00,  1.07s/it]


[Trial 1, Epoch 7] Kappa: 0.8216


  8%|▊         | 144/1768 [03:25<38:36,  1.43s/it]


[W 2026-05-13 21:43:49,010] Trial 2 failed with parameters: {'gamma': 2.5705579538204493, 'w_focal': 1.8607218154152545, 'w_ord': 1.9277662293290327, 'lr': 1.4446297571617738e-05, 'weight_decay': 9.996761016947173e-05} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/home/woshington/Projects/Doutorado/repo/.venv/lib/python3.12/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_165134/693868007.py", line 57, in objective
    training_step(model, optuna_train_loader, opt, device, loss_fn, scaler, grad_clip=GRAD_CLIP)
  File "/tmp/ipykernel_165134/1481531616.py", line 15, in training_step
    scaler.step(optimizer)
  File "/home/woshington/Projects/Doutorado/repo/.venv/lib/python3.12/site-packages/torch/amp/grad_scaler.py", line 465, in step
    retval = self._maybe_opt_step(optimizer, optimizer_state, *args, **kwargs)
             ^^^^^

KeyboardInterrupt: 

In [ ]:
# Importância dos hiperparâmetros
importances = optuna.importance.get_param_importances(study)
print('Importância dos hiperparâmetros:')
for param, imp in importances.items():
    print(f'  {param}: {imp:.4f}')

# Extrai melhores parâmetros
BEST_GAMMA        = best.params['gamma']
BEST_W_FOCAL      = best.params['w_focal']
BEST_W_ORD        = best.params['w_ord']
BEST_LR           = best.params['lr']
BEST_WEIGHT_DECAY = best.params['weight_decay']

print(f'\nMelhores hiperparâmetros:')
print(f'  gamma={BEST_GAMMA:.4f}, w_focal={BEST_W_FOCAL:.4f}, w_ord={BEST_W_ORD:.4f}')
print(f'  lr={BEST_LR:.2e}, weight_decay={BEST_WEIGHT_DECAY:.2e}')

## Dashboard e Visualizações Optuna

O estudo é persistido em SQLite — use as células abaixo para análise inline (Plotly)
ou inicie o dashboard web com o comando na última célula desta seção.

| Gráfico | O que mostra |
|---|---|
| `plot_optimization_history` | Evolução do Kappa ao longo dos trials |
| `plot_intermediate_values` | Valores intermediários (por época) com pruning |
| `plot_parallel_coordinate` | Correlação entre hiperparâmetros e objetivo |
| `plot_param_importances` | Importância relativa de cada hiperparâmetro |
| `plot_contour` | Superfície 2-D de pares de hiperparâmetros |
| `plot_slice` | Efeito marginal de cada hiperparâmetro |


In [ ]:
from optuna import visualization as optvis

fig = optvis.plot_optimization_history(study)
fig.update_layout(title='Histórico de Otimização — Kappa por Trial', height=450)
fig.show()


In [ ]:
fig = optvis.plot_intermediate_values(study)
fig.update_layout(title='Valores Intermediários por Época (trials com pruning)', height=450)
fig.show()


In [ ]:
fig = optvis.plot_parallel_coordinate(study, params=['gamma', 'w_focal', 'w_ord', 'lr', 'weight_decay'])
fig.update_layout(title='Coordenadas Paralelas — Hiperparâmetros vs Kappa', height=500)
fig.show()

In [ ]:
fig = optvis.plot_param_importances(study)
fig.update_layout(title='Importância dos Hiperparâmetros (fANOVA) — loss + otimizador', height=400)
fig.show()

In [ ]:
# Pares de hiperparâmetros com maior impacto (inclui lr e weight_decay)
for p1, p2 in [('lr', 'weight_decay'), ('gamma', 'w_focal'), ('gamma', 'w_ord'), ('w_focal', 'w_ord')]:
    fig = optvis.plot_contour(study, params=[p1, p2])
    fig.update_layout(title=f'Contorno: {p1} × {p2}', height=500)
    fig.show()

In [ ]:
fig = optvis.plot_slice(study, params=['gamma', 'w_focal', 'w_ord', 'lr', 'weight_decay'])
fig.update_layout(title='Efeito Marginal de Cada Hiperparâmetro', height=450)
fig.show()

### Dashboard Web

Inicia o servidor Optuna Dashboard na porta 8080.

In [ ]:
# Lança o Optuna Dashboard no navegador (porta 8080 por padrão)
# Instale antes: pip install optuna-dashboard
# Execute na célula ou no terminal:

print(f'Storage: {STUDY_DB}')
print(f'Study  : {STUDY_NAME}')
print()
print('Para abrir o dashboard, execute no terminal:')
print(f'  optuna-dashboard {STUDY_DB}')
print()
print('Ou diretamente desta célula (abre em background):')


In [ ]:
import subprocess, threading

def _run_dashboard():
    subprocess.run(['optuna-dashboard', STUDY_DB], check=False)

thread = threading.Thread(target=_run_dashboard, daemon=True)
thread.start()
print('Dashboard iniciado em http://127.0.0.1:8080')
print('(o processo fica em background; reinicie o kernel para parar)')


## Treino Completo com Melhores Hiperparâmetros

In [ ]:
train_dataset = PandasWithMilDataset(
    IMAGES_DIR, df_train, transforms=train_transforms,
    normalize=False, max_patches=MAX_PATCHES
)
valid_dataset = PandasWithMilDataset(
    IMAGES_DIR, df_val, transforms=val_transforms,
    normalize=False, max_patches=MAX_PATCHES
)
test_dataset  = PandasWithMilDataset(
    IMAGES_DIR, df_test, transforms=val_transforms,
    normalize=False, max_patches=MAX_PATCHES
)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
    sampler=RandomSampler(train_dataset),
    pin_memory=True, prefetch_factor=2, persistent_workers=True, drop_last=True,
)
valid_loader = DataLoader(
    valid_dataset, batch_size=BATCH_SIZE * 2, num_workers=NUM_WORKERS,
    sampler=SequentialSampler(valid_dataset),
    pin_memory=True, prefetch_factor=2, persistent_workers=True,
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE * 2, num_workers=NUM_WORKERS,
    shuffle=False, pin_memory=True, prefetch_factor=2, persistent_workers=True,
)

print(f'Train: {len(train_loader)} batches | Val: {len(valid_loader)} batches | Test: {len(test_loader)} batches')

In [ ]:
model = build_model(enable_checkpointing=True)

# torch.compile: JIT kernels otimizados para RTX 3060 (Ampere sm_86).
# mode='reduce-overhead' minimiza lançamentos de kernel — bom para loops pequenos.
# Aplicado apenas aqui (não no Optuna) para evitar custo de compilação por trial.
model = torch.compile(model, mode='reduce-overhead')

loss_function = BCEFocalOrdinalLoss(
    gamma=BEST_GAMMA,
    w_focal=BEST_W_FOCAL,
    w_ord=BEST_W_ORD,
)

optimizer = optim.AdamW(
    model.parameters(),
    lr=BEST_LR / WARMUP_FACTOR,
    weight_decay=BEST_WEIGHT_DECAY,
)

scheduler_cosine = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, N_EPOCHS - WARMUP_EPOCHS
)
scheduler = GradualWarmupScheduler(
    optimizer,
    multiplier=WARMUP_FACTOR,
    total_epoch=WARMUP_EPOCHS,
    after_scheduler=scheduler_cosine,
)

scaler = torch.amp.GradScaler()

print(f'Modelo: ViT-B/16 + MIL (GatedAttention) — compilado com torch.compile')
print(f'Perda : gamma={BEST_GAMMA:.4f}, w_focal={BEST_W_FOCAL:.4f}, w_ord={BEST_W_ORD:.4f}')
print(f'Opt   : AdamW | lr={BEST_LR:.2e} | weight_decay={BEST_WEIGHT_DECAY:.2e}')
print(f'AMP   : {AMP_DTYPE} | GRAD_CLIP={GRAD_CLIP} | BATCH={BATCH_SIZE} | PATCHES={MAX_PATCHES}')
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Params treináveis: {trainable:,} / {total:,}')

In [ ]:
best_kappa = 0.0
best_epoch = 0
no_improve = 0

history = {
    'train_loss':    [],
    'val_loss':      [],
    'val_acc':       [],
    'val_kappa':     [],
    'val_f1':        [],
    'val_recall':    [],
    'val_precision': [],
}

print('Iniciando treino completo...')
print('=' * 80)

for epoch in range(1, N_EPOCHS + 1):
    print(f'\nÉpoca {epoch}/{N_EPOCHS}')

    train_losses = training_step(model, train_loader, optimizer, device, loss_function, scaler, grad_clip=GRAD_CLIP)
    metrics      = validation_step(model, valid_loader, device, loss_function)

    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]

    history['train_loss'].append(np.mean(train_losses))
    for k in ['val_loss', 'val_acc', 'val_kappa', 'val_f1', 'val_recall', 'val_precision']:
        history[k].append(metrics[k])

    print(f'  Train Loss: {history["train_loss"][-1]:.5f}')
    print(f'  Val   Loss: {metrics["val_loss"]:.5f} | Acc: {metrics["val_acc"]*100:.2f}% | Kappa: {metrics["val_kappa"]:.4f} | F1: {metrics["val_f1"]:.4f}')
    print(f'  LR: {current_lr:.2e}')

    log_line = (f'epoch: {epoch} | lr: {current_lr:.2e} | '
                f'train_loss: {history["train_loss"][-1]:.5f} | '
                f'val_loss: {metrics["val_loss"]:.5f} | '
                f'val_acc: {metrics["val_acc"]:.4f} | '
                f'val_kappa: {metrics["val_kappa"]:.4f}\n')
    with open(LOG_PATH, 'a') as f:
        f.write(log_line)

    if metrics['val_kappa'] > best_kappa:
        best_kappa = metrics['val_kappa']
        best_epoch = epoch
        no_improve = 0
        torch.save(model.state_dict(), MODEL_PATH)
        print(f'  ✓ Melhor modelo salvo! Kappa: {best_kappa:.4f}')
    else:
        no_improve += 1
        print(f'  Sem melhora por {no_improve} época(s)')

    if no_improve >= PATIENCE:
        print(f'\nEarly stopping na época {epoch}. Melhor: época {best_epoch} (Kappa={best_kappa:.4f})')
        break

print('\nTreino concluído!')
print(f'Melhor Kappa de validação: {best_kappa:.4f} na época {best_epoch}')

## Curvas de Aprendizado

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

axes[0, 0].plot(history['train_loss'], label='Train Loss')
axes[0, 0].plot(history['val_loss'],   label='Val Loss')
axes[0, 0].set_title('Loss')
axes[0, 0].set_xlabel('Época')
axes[0, 0].legend()
axes[0, 0].grid(True)

axes[0, 1].plot(history['val_acc'], color='green', label='Val Accuracy')
axes[0, 1].set_title('Acurácia de Validação')
axes[0, 1].set_xlabel('Época')
axes[0, 1].legend()
axes[0, 1].grid(True)

axes[1, 0].plot(history['val_kappa'], color='orange', label='Val Kappa')
axes[1, 0].set_title('Kappa Quadrático de Validação')
axes[1, 0].set_xlabel('Época')
axes[1, 0].legend()
axes[1, 0].grid(True)

axes[1, 1].plot(history['val_f1'], color='red', label='Val F1')
axes[1, 1].set_title('F1 Macro de Validação')
axes[1, 1].set_xlabel('Época')
axes[1, 1].legend()
axes[1, 1].grid(True)

plt.tight_layout()
plt.savefig('logs/vit-base-mil-optuna-training.png', dpi=300, bbox_inches='tight')
plt.show()

## Avaliação no Conjunto de Teste

In [ ]:
model.load_state_dict(torch.load(MODEL_PATH, weights_only=True))
model.eval()

all_preds, all_targets = [], []

with torch.no_grad():
    for bag, mask, targets, _ in tqdm(test_loader, desc='Testing'):
        bag  = bag.to(device,  non_blocking=True)
        mask = mask.to(device, non_blocking=True)
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            out = model(bag, mask)
        probs       = torch.sigmoid(out['logits'])
        preds       = (probs > 0.5).sum(dim=1)
        targets_cls = targets.sum(dim=1).long()
        all_preds.append(preds.cpu())
        all_targets.append(targets_cls.cpu())

all_preds   = torch.cat(all_preds).numpy()
all_targets = torch.cat(all_targets).numpy()

test_acc   = accuracy_score(all_targets, all_preds)
test_kappa = cohen_kappa_score(all_targets, all_preds, weights='quadratic')
test_f1    = f1_score(all_targets, all_preds, average='macro', zero_division=0)

print('=' * 60)
print('RESULTADOS NO TESTE')
print('=' * 60)
print(f'Acurácia : {test_acc*100:.2f}%')
print(f'Kappa    : {test_kappa:.4f}')
print(f'F1 Macro : {test_f1:.4f}')
print('=' * 60)
print(f'Hiperparâmetros Optuna usados:')
print(f'  gamma={BEST_GAMMA:.4f}, w_focal={BEST_W_FOCAL:.4f}, w_ord={BEST_W_ORD:.4f}')

In [ ]:
cm = confusion_matrix(all_targets, all_preds)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

classes = ['ISUP 0', 'ISUP 1', 'ISUP 2', 'ISUP 3', 'ISUP 4', 'ISUP 5']

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm_norm, annot=True, fmt='.3f', cmap='Blues',
    xticklabels=classes, yticklabels=classes,
)
plt.title('Matriz de Confusão Normalizada — ViT-B/16 MIL')
plt.ylabel('Classe Verdadeira')
plt.xlabel('Classe Prevista')
plt.tight_layout()
plt.savefig('logs/vit-base-mil-optuna-confusion-matrix.png', dpi=300, bbox_inches='tight')
plt.show()